# Nyaya Darshana — Safe Legal-Model Training

1. Attach the original dataset containing `train.jsonl` and `validation.jsonl`.
2. Enable a GPU accelerator and internet access in Kaggle settings.
3. Run all cells.
4. Download `nyaya_model_release.zip` from the Output tab.

The adapter is exported before evaluation. A failed legal-quality gate never deletes it, but it must not be deployed until the evaluation report passes.


In [ ]:
# Install only when the required package is missing.
import importlib.util, subprocess, sys
packages = {'trl': 'trl', 'peft': 'peft', 'bitsandbytes': 'bitsandbytes', 'accelerate': 'accelerate', 'datasets': 'datasets'}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready:', ', '.join(packages))


In [ ]:
"""Data-audit and release-gate utilities for Nyaya Darshana model training.

This module intentionally has no third-party dependencies so that it can run
before GPU packages are installed and inside Kaggle notebook cells.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from hashlib import sha256
import json
from pathlib import Path
import re
from typing import Any, Iterable, Mapping, Sequence


FABRICATED_STATUTE_PATTERNS = (
    r"\bb\.?\s*p\.?\s*singh\s+penal\s+code\b",
    r"\bbordeau(?:x)?(?:[-\s]nariman)?\b",
    r"\bbharatiya\s+smriti\s+sanhita\b",
    r"\bbcriminal\s+procedure\s+code\b",
    r"\bbcrpc\b",
    r"\bmission\s+250\b",
)


@dataclass(frozen=True)
class LegalProbe:
    name: str
    question: str
    required_any: tuple[str, ...]
    forbidden: tuple[str, ...] = ()
    critical: bool = True


LEGAL_PROBES: tuple[LegalProbe, ...] = (
    LegalProbe(
        "ipc_successor",
        "Which statute replaced the Indian Penal Code, 1860? Give its complete name.",
        (r"\bbharatiya\s+nyaya\s+sanhita\b",),
        (r"\bbharatiya\s+nagarik\s+suraksha\s+sanhita\b",),
    ),
    LegalProbe(
        "crpc_successor",
        "Which statute replaced the Code of Criminal Procedure, 1973? Give its complete name.",
        (r"\bbharatiya\s+nagarik\s+suraksha\s+sanhita\b",),
        (r"\bbharatiya\s+nyaya\s+sanhita\s+(?:replaced|governs)\s+(?:the\s+)?(?:code\s+of\s+criminal\s+procedure|crpc)",),
    ),
    LegalProbe(
        "evidence_successor",
        "Which statute replaced the Indian Evidence Act, 1872? Give its complete name.",
        (r"\bbharatiya\s+sakshya\s+adhiniyam\b",),
        (r"\binformation\s+technology\s+act\b",),
    ),
    LegalProbe(
        "pocso_independent",
        "Did the Bharatiya Nyaya Sanhita repeal or replace the POCSO Act, 2012?",
        (r"\b(?:no|not|neither|remains|continues|separate|special)\b",),
        (
            r"\b(?:bns|bharatiya\s+nyaya\s+sanhita)\s+(?:has\s+)?(?:repealed|replaced|subsumed)\s+(?:the\s+)?pocso\b",
        ),
    ),
    LegalProbe(
        "bns_not_procedure",
        "Does the Bharatiya Nyaya Sanhita replace the Code of Criminal Procedure?",
        (r"\b(?:no|not|bnss|bharatiya\s+nagarik\s+suraksha\s+sanhita)\b",),
        (r"\byes\b.{0,80}\b(?:bns|bharatiya\s+nyaya\s+sanhita)\b",),
    ),
    LegalProbe(
        "retrospective_substantive_law",
        "A theft occurred on 29 June 2024, but the FIR was registered on 3 July 2024. Does BNS apply retrospectively merely because the FIR was registered after 1 July?",
        (r"\b(?:ipc|indian\s+penal\s+code|not\s+retrospective|article\s+20\s*\(?1\)?)\b",),
        (r"\bbns\s+(?:applies|must\s+apply)\s+retrospectively\b",),
    ),
    LegalProbe(
        "procedural_savings",
        "Which savings provisions should be considered when an IPC-era offence is investigated after the new criminal laws took effect?",
        (r"\b(?:358|531|savings|repeal)\b",),
        (),
        False,
    ),
    LegalProbe(
        "zero_fir",
        "Can information about a cognizable offence be recorded irrespective of territorial jurisdiction under BNSS?",
        (r"\b(?:zero\s+fir|irrespective\s+of\s+territorial\s+jurisdiction|173)\b",),
        (),
        False,
    ),
    LegalProbe(
        "pocso_age",
        "Under POCSO, can a 17-year-old child's apparent consent automatically defeat the application of the Act?",
        (r"\b(?:no|minor|under\s+18|below\s+18|child)\b",),
        (r"\bconsent\s+(?:automatically\s+)?(?:defeats|bars|negates)\s+pocso\b",),
    ),
    LegalProbe(
        "electronic_evidence",
        "Which new statute governs electronic evidence after the replacement of the Indian Evidence Act?",
        (r"\b(?:bharatiya\s+sakshya\s+adhiniyam|bsa)\b",),
        (r"\bbharatiya\s+nyaya\s+sanhita\s+governs\s+electronic\s+evidence\b",),
    ),
)


def _normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value.casefold()).strip()


def _matches(pattern: str, value: str) -> bool:
    return bool(re.search(pattern, value, flags=re.IGNORECASE | re.DOTALL))


def record_fingerprint(record: Mapping[str, Any]) -> str:
    text = "\n".join(
        _normalize_text(str(record.get(field, "")))
        for field in ("instruction", "input", "output")
    )
    return sha256(text.encode("utf-8")).hexdigest()


def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    source = Path(path)
    if not source.is_file():
        raise FileNotFoundError(f"Dataset file does not exist: {source}")
    records: list[dict[str, Any]] = []
    with source.open("r", encoding="utf-8") as handle:
        for line_number, raw in enumerate(handle, start=1):
            if not raw.strip():
                continue
            try:
                value = json.loads(raw)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {source}:{line_number}: {exc}") from exc
            if not isinstance(value, dict):
                raise ValueError(f"Expected a JSON object in {source}:{line_number}")
            records.append(value)
    if not records:
        raise ValueError(f"Dataset is empty: {source}")
    return records


def audit_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    split_name: str,
    minimum_records: int = 1,
) -> dict[str, Any]:
    errors: list[str] = []
    warnings: list[str] = []
    categories: dict[str, int] = {}
    fingerprints: set[str] = set()
    duplicate_count = 0

    if len(records) < minimum_records:
        errors.append(
            f"{split_name} contains {len(records)} records; at least {minimum_records} are required"
        )

    for index, record in enumerate(records, start=1):
        identifier = str(record.get("id", f"row-{index}"))
        instruction = str(record.get("instruction", "")).strip()
        answer = str(record.get("output", "")).strip()
        if not instruction or not answer:
            errors.append(f"{split_name}:{identifier} is missing instruction or output")
            continue

        category = str(record.get("category", "uncategorized"))
        categories[category] = categories.get(category, 0) + 1
        fingerprint = record_fingerprint(record)
        if fingerprint in fingerprints:
            duplicate_count += 1
        fingerprints.add(fingerprint)

        for pattern in FABRICATED_STATUTE_PATTERNS:
            if _matches(pattern, answer):
                errors.append(
                    f"{split_name}:{identifier} contains a known fabricated statute pattern: {pattern}"
                )

        if len(answer) < 20:
            warnings.append(f"{split_name}:{identifier} has a very short completion")

    if duplicate_count:
        warnings.append(f"{split_name} contains {duplicate_count} duplicate examples")
    if len(categories) == 1 and len(records) >= 20:
        warnings.append(f"{split_name} contains only one category")
    if categories and len(records) >= 20:
        dominant_category, dominant_count = max(categories.items(), key=lambda item: item[1])
        if dominant_count / len(records) > 0.70:
            warnings.append(
                f"{split_name} is imbalanced: {dominant_category!r} accounts for "
                f"{dominant_count / len(records):.0%} of examples"
            )

    return {
        "split": split_name,
        "records": len(records),
        "categories": dict(sorted(categories.items())),
        "duplicate_count": duplicate_count,
        "fingerprints": fingerprints,
        "errors": errors,
        "warnings": warnings,
        "passed": not errors,
    }


def audit_splits(
    train: Sequence[Mapping[str, Any]],
    validation: Sequence[Mapping[str, Any]],
    test: Sequence[Mapping[str, Any]] | None = None,
) -> dict[str, Any]:
    audits = [
        audit_dataset(train, split_name="train", minimum_records=50),
        audit_dataset(validation, split_name="validation", minimum_records=10),
    ]
    if test is not None:
        audits.append(audit_dataset(test, split_name="test", minimum_records=10))

    errors = [message for audit in audits for message in audit["errors"]]
    warnings = [message for audit in audits for message in audit["warnings"]]
    overlap: dict[str, int] = {}
    for left_index, left in enumerate(audits):
        for right in audits[left_index + 1 :]:
            shared = left["fingerprints"] & right["fingerprints"]
            label = f"{left['split']}__{right['split']}"
            overlap[label] = len(shared)
            if shared:
                errors.append(f"Dataset leakage: {label} share {len(shared)} identical examples")

    safe_audits = []
    for audit in audits:
        safe_audits.append({key: value for key, value in audit.items() if key != "fingerprints"})

    return {
        "splits": safe_audits,
        "overlap": overlap,
        "errors": errors,
        "warnings": warnings,
        "passed": not errors,
    }


def score_answer(probe: LegalProbe, answer: str) -> dict[str, Any]:
    normalized = answer.strip()
    missing_required = not any(_matches(pattern, normalized) for pattern in probe.required_any)
    fabricated = [
        pattern
        for pattern in FABRICATED_STATUTE_PATTERNS
        if _matches(pattern, normalized)
    ]
    forbidden = [pattern for pattern in probe.forbidden if _matches(pattern, normalized)]
    passed = bool(normalized) and not missing_required and not fabricated and not forbidden
    return {
        "name": probe.name,
        "question": probe.question,
        "answer": normalized,
        "critical": probe.critical,
        "passed": passed,
        "missing_required": missing_required,
        "fabricated_patterns": fabricated,
        "forbidden_patterns": forbidden,
    }


def evaluate_answers(
    answers: Mapping[str, str],
    *,
    minimum_accuracy: float = 0.9,
    probes: Iterable[LegalProbe] = LEGAL_PROBES,
) -> dict[str, Any]:
    results = [score_answer(probe, answers.get(probe.name, "")) for probe in probes]
    passed_count = sum(item["passed"] for item in results)
    accuracy = passed_count / len(results) if results else 0.0
    critical_failures = [item["name"] for item in results if item["critical"] and not item["passed"]]
    hallucinations = [
        item["name"]
        for item in results
        if item["fabricated_patterns"] or item["forbidden_patterns"]
    ]
    return {
        "total": len(results),
        "passed": passed_count,
        "accuracy": round(accuracy, 4),
        "minimum_accuracy": minimum_accuracy,
        "critical_failures": critical_failures,
        "hallucinations": hallucinations,
        "release_ready": accuracy >= minimum_accuracy and not critical_failures and not hallucinations,
        "results": results,
    }


def probe_manifest() -> list[dict[str, Any]]:
    return [asdict(probe) for probe in LEGAL_PROBES]


In [ ]:
"""Safe, resumable Kaggle training for the Nyaya Darshana legal adapter.

Run on a Kaggle notebook with a GPU and train.jsonl / validation.jsonl attached.
The script saves the final adapter before evaluation and never loads a second
copy of the base model into the same GPU process.
"""

from __future__ import annotations

import gc
import json
import os
from dataclasses import dataclass
from pathlib import Path
import shutil
import sys
import time
from typing import Any


os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


try:
    from model_quality import LEGAL_PROBES, audit_splits, evaluate_answers, read_jsonl
except ImportError:
    try:
        from training.model_quality import LEGAL_PROBES, audit_splits, evaluate_answers, read_jsonl
    except ImportError:
        from __main__ import LEGAL_PROBES, audit_splits, evaluate_answers, read_jsonl


MODEL_NAME = os.getenv("NYAYA_BASE_MODEL", "unsloth/Meta-Llama-3.1-8B-Instruct")
WORKING_ROOT = Path(os.getenv("NYAYA_WORKING_ROOT", "/kaggle/working"))
if not WORKING_ROOT.is_dir():
    WORKING_ROOT = Path.cwd()
OUTPUT_ROOT = WORKING_ROOT / "nyaya_model_release"
ADAPTER_DIR = OUTPUT_ROOT / "adapter"
CHECKPOINT_DIR = WORKING_ROOT / "nyaya_training_checkpoints"
REPORT_DIR = OUTPUT_ROOT / "reports"
ARCHIVE_PATH = WORKING_ROOT / "nyaya_model_release.zip"
HF_TOKEN = os.getenv("HF_TOKEN")
MINIMUM_ACCURACY = float(os.getenv("NYAYA_MINIMUM_ACCURACY", "0.90"))
MAX_LENGTH = int(os.getenv("NYAYA_MAX_LENGTH", "768"))
MAX_STEPS = int(os.getenv("NYAYA_MAX_STEPS", "-1"))
SYSTEM_PROMPT = (
    "You are Nyaya Darshana, a precise Indian legal assistant. "
    "Use only genuine statutory names and provisions. "
    "IPC was replaced by Bharatiya Nyaya Sanhita (BNS); "
    "CrPC was replaced by Bharatiya Nagarik Suraksha Sanhita (BNSS); "
    "the Indian Evidence Act was replaced by Bharatiya Sakshya Adhiniyam (BSA). "
    "POCSO remains an independent special statute. "
    "Never invent a statute, section number, date, or judicial decision. "
    "If authority is uncertain, acknowledge that uncertainty."
)


def announce(message: str) -> None:
    print(f"[NYAYA] {message}", flush=True)


def resolve_dataset(filename: str, environment_name: str, *, required: bool) -> Path | None:
    override = os.getenv(environment_name)
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override))
    candidates.extend(
        [
            Path.cwd() / filename,
            Path.cwd() / "training" / filename,
            WORKING_ROOT / filename,
            Path("/content") / filename,
        ]
    )
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(sorted(kaggle_input.rglob(filename)))

    for candidate in candidates:
        if candidate.is_file() and candidate.stat().st_size > 0:
            announce(f"Using {filename}: {candidate}")
            return candidate
    if required:
        raise FileNotFoundError(
            f"Could not locate a non-empty {filename}. Attach the original Kaggle dataset "
            f"or set {environment_name} to its absolute path."
        )
    return None


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def directory_size_mb(path: Path) -> float:
    return round(
        sum(item.stat().st_size for item in path.rglob("*") if item.is_file()) / 1024**2,
        2,
    )


def save_release_archive() -> Path:
    archive = Path(shutil.make_archive(str(ARCHIVE_PATH.with_suffix("")), "zip", OUTPUT_ROOT))
    announce(f"Durable release archive: {archive} ({archive.stat().st_size / 1024**2:.2f} MB)")
    return archive


def build_chat_prompt(tokenizer: Any, question: str, answer: str | None = None) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer})
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def prepare_text_record(tokenizer: Any, record: dict[str, Any]) -> dict[str, str]:
    question = "\n".join(
        value for value in (str(record.get("instruction", "")).strip(), str(record.get("input", "")).strip()) if value
    )
    return {"text": build_chat_prompt(tokenizer, question, str(record["output"]).strip())}


def tokenize_completion_only(tokenizer: Any, record: dict[str, Any]) -> dict[str, list[int]]:
    question = "\n".join(
        value
        for value in (
            str(record.get("instruction", "")).strip(),
            str(record.get("input", "")).strip(),
        )
        if value
    )
    prompt = build_chat_prompt(tokenizer, question)
    completion = str(record["output"]).strip() + tokenizer.eos_token
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]

    if len(completion_ids) >= MAX_LENGTH:
        completion_ids = completion_ids[: MAX_LENGTH - 1]
    allowed_prompt = max(1, MAX_LENGTH - len(completion_ids))
    prompt_ids = prompt_ids[-allowed_prompt:]
    input_ids = prompt_ids + completion_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + completion_ids,
    }


@dataclass
class CompletionOnlyCollator:
    tokenizer: Any

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, Any]:
        import torch

        maximum_length = max(len(item["input_ids"]) for item in features)
        pad_token = self.tokenizer.pad_token_id
        inputs: list[list[int]] = []
        masks: list[list[int]] = []
        labels: list[list[int]] = []
        for item in features:
            pad_count = maximum_length - len(item["input_ids"])
            inputs.append(item["input_ids"] + [pad_token] * pad_count)
            masks.append(item["attention_mask"] + [0] * pad_count)
            labels.append(item["labels"] + [-100] * pad_count)
        return {
            "input_ids": torch.tensor(inputs, dtype=torch.long),
            "attention_mask": torch.tensor(masks, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def adapter_weight_path() -> Path:
    candidates = list(ADAPTER_DIR.glob("adapter_model.safetensors")) + list(ADAPTER_DIR.glob("adapter_model.bin"))
    if not candidates:
        raise RuntimeError(f"Training finished but no adapter weights were saved in {ADAPTER_DIR}")
    candidate = candidates[0]
    if candidate.stat().st_size < 1024 * 1024:
        raise RuntimeError(f"Adapter weights are unexpectedly small: {candidate.stat().st_size} bytes")
    return candidate


def find_latest_checkpoint() -> str | None:
    if not CHECKPOINT_DIR.is_dir():
        return None
    checkpoints = [item for item in CHECKPOINT_DIR.glob("checkpoint-*") if item.is_dir()]
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda item: int(item.name.rsplit("-", 1)[-1]))
    return str(checkpoints[-1])


def generate_answer(model: Any, tokenizer: Any, question: str) -> str:
    import torch

    prompt = build_chat_prompt(tokenizer, question)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.inference_mode():
        result = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = result[0][inputs["input_ids"].shape[1] :]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    del inputs, generated, result
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return answer


def main() -> dict[str, Any]:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    train_path = resolve_dataset("train.jsonl", "NYAYA_TRAIN_FILE", required=True)
    validation_path = resolve_dataset("validation.jsonl", "NYAYA_VALIDATION_FILE", required=True)
    test_path = resolve_dataset("test.jsonl", "NYAYA_TEST_FILE", required=False)
    assert train_path is not None and validation_path is not None

    train_records = read_jsonl(train_path)
    validation_records = read_jsonl(validation_path)
    test_records = read_jsonl(test_path) if test_path else None
    dataset_report = audit_splits(train_records, validation_records, test_records)
    write_json(REPORT_DIR / "dataset_audit.json", dataset_report)
    if not dataset_report["passed"]:
        raise RuntimeError("Dataset audit failed: " + "; ".join(dataset_report["errors"]))
    announce(
        f"Dataset audit passed: train={len(train_records)}, validation={len(validation_records)}, "
        f"test={len(test_records or [])}"
    )

    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback
    from trl import SFTConfig, SFTTrainer

    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required. Enable a Kaggle GPU accelerator before running.")

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    announce(f"GPU: {torch.cuda.get_device_name(0)}; visible GPUs: {torch.cuda.device_count()}")
    announce(f"Base model: {MODEL_NAME}")

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization,
        device_map={"": 0},
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(
        model,
        LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.10,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        ),
    )

    train_dataset = Dataset.from_list(train_records).map(
        lambda record: tokenize_completion_only(tokenizer, record),
        remove_columns=list(train_records[0].keys()),
    )
    validation_dataset = Dataset.from_list(validation_records).map(
        lambda record: tokenize_completion_only(tokenizer, record),
        remove_columns=list(validation_records[0].keys()),
    )
    data_collator = CompletionOnlyCollator(tokenizer=tokenizer)
    example_labels = train_dataset[0]["labels"]
    active_tokens = sum(value != -100 for value in example_labels)
    if not active_tokens:
        raise RuntimeError("Completion-only masking removed all supervised answer tokens")
    announce(f"Completion-only supervision active: {active_tokens} answer tokens in the first example")

    options: dict[str, Any] = {
        "output_dir": str(CHECKPOINT_DIR),
        "num_train_epochs": 2,
        "per_device_train_batch_size": 1,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "gradient_checkpointing": True,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.08,
        "weight_decay": 0.08,
        "logging_steps": 10,
        "eval_strategy": "steps",
        "eval_steps": 50,
        "save_strategy": "steps",
        "save_steps": 50,
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "optim": "paged_adamw_8bit",
        "max_length": MAX_LENGTH,
        "packing": False,
        "dataset_kwargs": {"skip_prepare_dataset": True},
        "report_to": "none",
        "seed": 42,
        "fp16": False,
        "bf16": False,
    }
    if MAX_STEPS > 0:
        options["max_steps"] = MAX_STEPS

    training_config = SFTConfig(**options)
    training_config._n_gpu = 1
    trainer = SFTTrainer(
        model=model,
        args=training_config,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    resume_from = find_latest_checkpoint()
    if resume_from:
        announce(f"Resuming from existing checkpoint: {resume_from}")

    started = time.time()
    training_result = trainer.train(resume_from_checkpoint=resume_from)
    metrics = trainer.evaluate()
    elapsed = round(time.time() - started, 2)

    announce("Saving final adapter and tokenizer before any inference evaluation")
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
    tokenizer.save_pretrained(ADAPTER_DIR)
    weight_path = adapter_weight_path()

    training_report = {
        "base_model": MODEL_NAME,
        "adapter_path": str(ADAPTER_DIR),
        "adapter_weight_file": weight_path.name,
        "adapter_size_mb": directory_size_mb(ADAPTER_DIR),
        "elapsed_seconds": elapsed,
        "global_step": training_result.global_step,
        "training_loss": float(training_result.training_loss),
        "validation_loss": float(metrics.get("eval_loss", 0.0)),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "release_ready": False,
        "evaluation_status": "pending",
    }
    write_json(REPORT_DIR / "training_report.json", training_report)
    save_release_archive()

    announce("Evaluating legal correctness using the existing model; no second base model is loaded")
    trainer.model.eval()
    trainer.model.config.use_cache = True
    answers: dict[str, str] = {}
    for probe in LEGAL_PROBES:
        answer = generate_answer(trainer.model, tokenizer, probe.question)
        answers[probe.name] = answer
        announce(f"Probe {probe.name}: {answer[:180]}")

    quality_report = evaluate_answers(answers, minimum_accuracy=MINIMUM_ACCURACY)
    write_json(REPORT_DIR / "legal_evaluation.json", quality_report)
    training_report["release_ready"] = quality_report["release_ready"]
    training_report["evaluation_status"] = "passed" if quality_report["release_ready"] else "failed"
    training_report["legal_accuracy"] = quality_report["accuracy"]
    write_json(REPORT_DIR / "training_report.json", training_report)
    save_release_archive()

    announce(
        f"Training complete. Accuracy={quality_report['accuracy']:.0%}; "
        f"release_ready={quality_report['release_ready']}; adapter={weight_path}"
    )
    if not quality_report["release_ready"]:
        announce(
            "The adapter was preserved, but deployment is blocked: "
            + ", ".join(quality_report["critical_failures"] or ["minimum accuracy not met"])
        )

    del trainer, model, train_dataset, validation_dataset
    gc.collect()
    torch.cuda.empty_cache()
    return {"training": training_report, "quality": quality_report, "dataset": dataset_report}


if __name__ == "__main__":
    main()
